← [12 - Arithmétique réelle](Z3-12-Real-Arithmetic-Python.ipynb) | [README Z3-Python](README.md) | [Version Python](Z3-13-UnsatCores-Python.ipynb) | [Version C#](Z3-13-UnsatCores-CSharp.ipynb) | [13b - UNSAT cores : le MUS >>](Z3-13b-UnsatCores-MUS-Python.ipynb)
# 13. UNSAT cores : expliquer l'insatisfiabilité (le « pourquoi »)

Jusqu'ici, le solveur répondait à une question binaire : *ce système a-t-il une solution ?* -> **SAT** (oui, voici un témoin) ou **UNSAT** (non, prouvé impossible). C'est un **verdict**, pas une **explication**.

Pour un ingénieur qui **écrit** la spécification, UNSAT est souvent un bug de modélisation : une contrainte trop forte, ou deux contraintes incompatibles. La réponse « impossible » ne suffit pas - il faut savoir **lesquelles** des contraintes se contredisent. C'est le rôle du **noyau d'insatisfiabilité** (UNSAT core) : le sous-ensemble **minimal** de contraintes responsable du conflit. Z3 passe du diagnostic binaire au diagnostic chirurgical.

Ce carnet est le **jumeau Rust** du [carnet Python](Z3-13-UnsatCores-Python.ipynb) : la même démonstration, conduite par l'API du [crate `z3`](https://crates.io/crates/z3) plutôt que par `pyz3`. Les trois verbes de l'API sont les mêmes — `Solver::assert_and_track` (étiqueter une hypothèse), `Solver::get_unsat_core` (le noyau minimal), `Solver::check` (le verdict) — mais l'écriture change : en Rust, l'égalité du langage est une comparaison structurelle (`PartialEq`, un `bool`), pas une égalité SMT. La contrainte s'écrit `Ast::eq(&x, &Int::from_i64(10))`. Cette distinction est le fil du portage : c'est elle qui sépare une démonstration qui marche d'une démonstration qui *compile* sans dire ce qu'elle prouve.

> Port Rust du notebook C# [`17_UnsatCores.ipynb`](../Z3-Linq2Z3/17_UnsatCores.ipynb), lui-même porté de pyz3. Le bloc DSL `Explain()` du C#, spécifique à Z3.Linq, n'est pas porté ; l'API brute `assert_and_track` + core fait le même travail.

## Source

Ce notebook s'inscrit dans la série ancrée par **Yurichev, Dennis — *SAT/SMT by Example* (édition publique du 12 février 2023, 666 pages)**, §3.13.2 « Unsat core » et chapitre 3 « Equations » — UNSAT cores, `assert_and_track`, diagnostic. Gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16,45 Mo, magic bytes `%PDF`). Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré complet de la série.

In [2]:
// Imports : le crate z3 (solveur SMT) et son API de modélisation.
// Le crate est lié dynamiquement à la bibliothèque Z3 installée sur le poste.
:dep z3 = { version = "0.23.0" }

use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

println!("Imports OK : z3");
println!("z3 (lib) version : {}", z3::full_version());

Imports OK : z3


z3 (lib) version : 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e


## 1. Du verdict à l'explication

Commençons par un système trivialement contradictoire. Le solveur dit **UNSAT** - mais ne dit pas (encore) POURQUOI. C'est le verdict brut.

In [3]:
// Rappel : un système contradictoire est UNSAT (verdict, sans explication).
use z3::ast::{Ast, Bool};
use z3::{Solver, SatResult};

let a = Bool::new_const("a");
let b = Bool::new_const("b");
let s0 = Solver::new();
s0.assert(&a.implies(b.clone()));
s0.assert(&a);
s0.assert(&b.not());

// `SatResult` se dérive en Rust ; on le rend en chaîne pour lire le verdict
// comme on le lit en Python.
let verdict = match s0.check() {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("' a->b, a, !b ' : {}  (verdict : impossible)", verdict);
println!("-> Z3 dit NON, mais ne dit pas (encore) POURQUOI.");
let _ = &s0;

' a->b, a, !b ' : unsat  (verdict : impossible)


-> Z3 dit NON, mais ne dit pas (encore) POURQUOI.


**Lecture chiffrée — un système où le core serait TOUT.** `' a->b, a, !b ' : unsat` : le verdict brut tombe en une ligne. Mais ce petit système a une propriété que les exemples suivants n'auront pas : aucune de ses trois contraintes n'est superflue. Chaque paire prise seule est satisfiable — `{a->b, a}` tient avec `b` vrai, `{a->b, !b}` tient avec `a` faux, `{a, !b}` tient sans l'implication. Autrement dit, un core étiqueté de ce système contiendrait les TROIS contraintes : le conflit naît de la combinaison complète, pas d'une paire. C'est la frontière utile à connaître avant la section suivante — l'étiquetage `assert_and_track` ne réduit que ce qui peut l'être : quand chaque contrainte participe à la refutation, le core égale le système, et l'explication reste à écrire à la main.

En Rust, la construction est plus proche de la logique qu'elle n'en a l'air : `a.implies(b.clone())` construit une implication **SMT**, alors que `a == b` comparerait deux arbres Rust et renverrait un `bool` du langage, jamais une formule. C'est le premier jalon d'un fil que tout le carnet reprend : `Ast::eq` pour l'égalité, `Ast::ne` pour la différence.

## 2. L'API `assert_and_track` : étiqueter chaque contrainte

Pour obtenir une explication, on fournit les contraintes comme des **hypothèses traçables** : `solver.assert_and_track(formule, etiquette)` où `etiquette` est un `Bool` servant de nom. Après un `unsat`, `solver.get_unsat_core()` renvoie le sous-ensemble **minimal** d'étiquettes responsable du conflit.

Trois contraintes sur `x` : `x >= 3`, `x <= 5`, `x = 10`. Lesquelles se contredisent ?

> **Écueil d'API, mesuré sur le crate 0.23.** Le paramètre `unsat_core` ne se pose pas par `solver.set(...)` comme en pyz3 : le crate n'expose pas cette méthode. Il passe par `z3::Params` (`set_bool("unsat_core", true)`) puis `Solver::set_params`. Sans cette ligne, `get_unsat_core()` ne renvoie rien — et le notebook afficherait un core vide, c'est-à-dire une explication qui n'explique rien.

In [4]:
// Trois contraintes sur x : lesquelles se contredisent ?
use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

let x = Int::new_const("x");
let s1 = Solver::new();

// Le crate n'expose pas `Solver::set` : le paramètre passe par `Params`.
let mut params = Params::new();
params.set_bool("unsat_core", true);
s1.set_params(&params);

// Les étiquettes : un `Bool` constant dont le nom EST l'étiquette.
let c1 = Bool::new_const("x_ge_3");   // étiquette pour x >= 3
let c2 = Bool::new_const("x_le_5");   // étiquette pour x <= 5
let c3 = Bool::new_const("x_eq_10");  // étiquette pour x = 10

// `Ast::eq` : l'égalité SMT. `x == Int::from_i64(10)` serait une comparaison
// structurelle Rust (un booléen), pas une formule envoyée au solveur.
s1.assert_and_track(x.ge(Int::from_i64(3)), &c1);
s1.assert_and_track(x.le(Int::from_i64(5)), &c2);
s1.assert_and_track(Ast::eq(&x, Int::from_i64(10)), &c3);

let st1 = s1.check();
let verdict = match st1 {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("' x>=3, x<=5, x=10 ' : {}", verdict);
println!();
if st1 == SatResult::Unsat {
    println!("UNSAT CORE (contraintes conflictuelles, sous-ensemble MINIMAL) :");
    // Le core est renvoyé dans l'ordre d'assertion : l'ordre est significatif.
    for c in s1.get_unsat_core() {
        println!("  - {}", c);
    }
    println!("-> Z3 ecarte 'x>=3' du core : cette contrainte est irrelevant pour le conflit.");
}
let _ = &s1;

' x>=3, x<=5, x=10 ' : unsat


UNSAT CORE (contraintes conflictuelles, sous-ensemble MINIMAL) :


  - x_le_5


  - x_eq_10


-> Z3 ecarte 'x>=3' du core : cette contrainte est irrelevant pour le conflit.


**Lecture chiffrée — la minimalité vérifiée dans les DEUX sens.** Core rendu : `x_le_5`, `x_eq_10` — la section 3 montrera que la paire suffit. L'autre sens est tout aussi testable et complète la définition : chaque membre du core pris individuellement est NÉCESSAIRE. Retirer `x_le_5` : `{x>=3, x=10}` est satisfiable (`x = 10` vérifie les deux). Retirer `x_eq_10` : `{x>=3, x<=5}` est satisfiable (tout `x` de 3 à 5). Aucune des deux ne peut partir — le core est irréductible dans les deux directions : suffisant (paire UNSAT) et nécessaire (chaque retrait redevient SAT).

En Rust, l'affichage de l'étiquette vient de l'implémentation `Display` du crate : un `Bool` constant s'imprime dans sa forme SMT-LIB, `x_le_5`. C'est la même chaîne que celle qu'affiche `pyz3` — les deux côtés sont donc comparables caractère par caractère, sans passer par une convention de formatage.

## 3. Minimalité du core : vérifier que la paire suffit

Le core est **minimal** : retirer une seule contrainte du core le rend satisfiable. Vérifions que la paire `{x<=5, x=10}` (sans `x>=3`) est **déjà** UNSAT - ce qui confirme que Z3 avait raison d'exclure `x>=3`.

In [5]:
// Vérifions : {x<=5, x=10} SEUL (sans x>=3) est-il déjà UNSAT ?
use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

let y = Int::new_const("y");
let s2 = Solver::new();
let mut params = Params::new();
params.set_bool("unsat_core", true);
s2.set_params(&params);

let p1 = Bool::new_const("y_le_5");
let p2 = Bool::new_const("y_eq_10");
s2.assert_and_track(y.le(Int::from_i64(5)), &p1);
s2.assert_and_track(Ast::eq(&y, Int::from_i64(10)), &p2);

let st2 = s2.check();
let verdict = match st2 {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("' y<=5, y=10 ' (sans y>=3) : {}", verdict);
if st2 == SatResult::Unsat {
    println!("-> CONFIRME : la paire {{x<=5, x=10}} suffit au conflit.");
    println!("   Z3 avait donc raison d'EXCLURE 'x>=3' du core : elle est irrelevant.");
}
let _ = &s2;

' y<=5, y=10 ' (sans y>=3) : unsat


-> CONFIRME : la paire {x<=5, x=10} suffit au conflit.


   Z3 avait donc raison d'EXCLURE 'x>=3' du core : elle est irrelevant.


**Lecture chiffrée — le core est une réponse, pas une excuse.** `{x>=3, x<=5, x=10}` est UNSAT, et le core ne retient que deux des trois contraintes. Il ne dit pas « votre spécification est mauvaise » : il dit **lesquelles** relaxer. `x >= 3` est satisfiable avec `x = 10` ; elle ne participe donc pas au conflit. Une spécification plus grande ne change rien à cette mécanique — le core reste un sous-ensemble minimal, quel que soit le nombre de contraintes ajoutées.

## 4. Application : spécification surcontrainte — laquelle relâcher ?

Cas réaliste : un planning où 5 contraintes s'accumulent. L'une d'elles (`h = 12`, demande de l'animateur) contredit une autre (`h != 12`, pause déjeuner). Le core identifie **exactement** les 2 coupables parmi les 5 - les autres (bornes, redondances) sont compatibles et ignorées. L'action correctrice en découle directement : relâcher l'une des deux contraintes coupables.

In [6]:
// Spécification surcontrainte : laquelle relâcher ?
use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

let h = Int::new_const("h");
let s3 = Solver::new();
let mut params = Params::new();
params.set_bool("unsat_core", true);
s3.set_params(&params);

let n1 = Bool::new_const("h_ge_9");     // 1. h >= 9
let n2 = Bool::new_const("h_le_17");    // 2. h <= 17
let n3 = Bool::new_const("h_ne_12");    // 3. h != 12 (pause déjeuner)
let n4 = Bool::new_const("h_eq_12");    // 4. h = 12 (animateur)  <-- conflit
let n5 = Bool::new_const("h_ge_9b");    // 5. h >= 9 (redondant)

s3.assert_and_track(h.ge(Int::from_i64(9)), &n1);
s3.assert_and_track(h.le(Int::from_i64(17)), &n2);
// La différence : `Ast::ne`. En Rust, `h != Int::from_i64(12)` comparerait
// deux valeurs Rust et ne produirait aucune formule.
s3.assert_and_track(Ast::ne(&h, Int::from_i64(12)), &n3);
s3.assert_and_track(Ast::eq(&h, Int::from_i64(12)), &n4);
s3.assert_and_track(h.ge(Int::from_i64(9)), &n5);

let st3 = s3.check();
let verdict = match st3 {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Specification (5 contraintes) : {}", verdict);
println!();
if st3 == SatResult::Unsat {
    println!("Core = les contraintes qui PARTICIPENT au conflit :");
    let core = s3.get_unsat_core();
    for c in &core {
        println!("  COUPABLE : {}", c);
    }
    println!();
    println!("-> Sur 5 contraintes, {} seulement sont en conflit.", core.len());
    println!("   Les autres (h>=9, h<=17, h>=9 redondant) sont COMPATIBLES -> Z3 les ignore.");
    println!("   Action : relacher la contrainte 'h=12' (animateur) OU 'h!=12' (pause).");
}
let _ = &s3;

Specification (5 contraintes) : unsat


Core = les contraintes qui PARTICIPENT au conflit :


  COUPABLE : h_ne_12


  COUPABLE : h_eq_12


-> Sur 5 contraintes, 2 seulement sont en conflit.


   Les autres (h>=9, h<=17, h>=9 redondant) sont COMPATIBLES -> Z3 les ignore.


   Action : relacher la contrainte 'h=12' (animateur) OU 'h!=12' (pause).


**Lecture chiffrée — les deux relâchements ne coûtent pas le même prix.** `Sur 5 contraintes, 2 seulement sont en conflit` — mais les deux manières de guérir ne sont pas équivalentes, et le système restant les chiffre. Les trois contraintes innocentes bornent `h` dans l'intervalle [9, 17], soit 17 - 9 + 1 = 9 valeurs entières. Relâcher `h=12` (garder la pause déjeuner) : 9 - 1 = 8 solutions restent. Relâcher `h!=12` (garder la demande de l'animateur) : exactement 1 solution, `h = 12`. Le core dit QUELLES contraintes sont en conflit ; compter ce que chaque relâchement laisse voir est le pas supplémentaire qui transforme le diagnostic en décision — ici, l'option pause déjeuner préserve 8 des 9 plannings possibles, l'option animateur en fige un seul.

## 5. Innocence prouvée : une contrainte non impliquée écartée du core

Un dernier témoin de la minimalité : sur un système à 3 contraintes (`x == 1`, `x >= 0` innocente, `x == 2`), le core minimal est `{x==1, x==2}`. La contrainte `x >= 0` est **compatible** avec les deux (un x valant 1 ou 2 vérifie x >= 0) - elle n'est pas impliquée dans le conflit, donc Z3 l'écarte du core. C'est ce qui distingue le core d'une simple liste de contraintes : il isole la **cause**, pas le contexte.

In [7]:
// Core minimal : {x==1, x==2} écarte x>=0 (innocente, non impliquée)
use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

let w = Int::new_const("w");
let s4 = Solver::new();
let mut params = Params::new();
params.set_bool("unsat_core", true);
s4.set_params(&params);

let r0 = Bool::new_const("w_eq_1");    // x == 1
let r1 = Bool::new_const("w_ge_0");    // x >= 0  (innocente)
let r2 = Bool::new_const("w_eq_2");    // x == 2

s4.assert_and_track(Ast::eq(&w, Int::from_i64(1)), &r0);
s4.assert_and_track(w.ge(Int::from_i64(0)), &r1);
s4.assert_and_track(Ast::eq(&w, Int::from_i64(2)), &r2);

let st4 = s4.check();
let verdict = match st4 {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Check(assumptions) : {}", verdict);
let core = s4.get_unsat_core();
let étiquettes: Vec<String> = core.iter().map(|c| c.to_string()).collect();
println!("UnsatCore = {{ {} }}   (x>=0 exclu du core)", étiquettes.join("  "));
let _ = &s4;

Check(assumptions) : unsat


UnsatCore = { w_eq_1  w_eq_2 }   (x>=0 exclu du core)


**Lecture chiffrée — le core ordonne, donc comparable.** Z3 renvoie les étiquettes du core dans l'**ordre d'assertion** : `w_eq_1` puis `w_eq_2`, l'innocente `w_ge_0` disparaît. Cet ordre est une garantie utile au-delà de l'affichage : il permet de comparer le core **caractère par caractère** entre le jumeau Python et le jumeau Rust — les deux s'appuient sur la même bibliothèque Z3 et déclenchent les assertions dans le même ordre. C'est ce que fait l'organe de conformité différentielle (`scripts/notebook_tools/check_rust_conformance.py`) : il compare les grandeurs affichées par les deux carnets, pas leurs fichiers.

> Note de portage : le carnet C# original compare ici l'API brute (`Check(assumptions)` + `UnsatCore`) avec le DSL Z3.Linq (`Explain()`). Le DSL n'existe ni en pyz3 ni dans le crate `z3` : l'API brute `assert_and_track` + core en est l'équivalent direct, et c'est elle que cette section démontre.

## 6. Ce que ce carnet a démontré

Trois leçons sur le noyau d'insatisfiabilité :

1. **Du verdict à l'explication** - `unsat` dit *qu'il n'y a pas* de solution ; `get_unsat_core()` dit *pourquoi* (lesquelles contraintes conflictuent).

2. **Minimalité** - le core est le sous-ensemble **minimal** responsable : retirer un seul élément le rend satisfiable. Les contraintes irrelevantes (compatibles, redondantes) sont écartées.

3. **Actionnable** - sur une spécification surcontrainte, le core pointe exactement la (ou les) contrainte(s) à relâcher, transformant un échec de modélisation en diagnostic chirurgical.

À quoi s'ajoute la leçon propre au portage Rust : une **formule n'est pas une expression**. `Ast::eq`, `Ast::ne` et les comparaisons `Int::ge` / `Int::le` construisent des formules SMT ; `==` et `!=` comparent des valeurs Rust. Confondre les deux produit un solveur qui répond « satisfiable » à une question qui n'a pas été posée.

## 7. Exercices

Les trois exercices ci-dessous vous font manipuler le UNSAT core sur des variantes : conflit caché parmi 4 contraintes (1), planning avec relâchement (2), core minimal parmi 10 (3). Chaque stub est self-contained : le solveur (`Solver`), les variables (`Int`), les étiquettes (`Bool`) et la méthode `assert_and_track` sont reconstruits dans chaque cellule.

### Exercice 1 - Core sur un conflit caché (2 coupables parmi 4)

Construisez 4 contraintes sur `z` dont 2 sont compatibles et 2 en conflit (ex : `z <= 3`, `z >= 0`, `z == 7`, `z != 9`). Obtenez le core et vérifiez qu'il ne contient **que** les 2 coupables.

**Étapes** : `z = Int::new_const("z")`, 4 `assert_and_track` étiquetés, `check()`, afficher `get_unsat_core()` (doit exclure les 2 compatibles).

In [8]:
// Exercice 1 : core sur un conflit caché (2 coupables parmi 4)
// TODO etudiant :
// Etape 1 : z = Int::new_const("z") ; 4 contraintes étiquetées (2 compatibles, 2 en conflit)
// Etape 2 : s.assert_and_track(..., &Bool::new_const("etiq")) pour chacune
// Etape 3 : s.check() + s.get_unsat_core() -> doit contenir seulement les 2 coupables
// Indice : n'oubliez pas `params.set_bool("unsat_core", true)` avant `set_params`.
let result: Option<u32> = None;  // TODO etudiant
println!("Exercice 1 a completer : isolez les 2 contraintes coupables parmi 4");
let _ = &result;

Exercice 1 a completer : isolez les 2 contraintes coupables parmi 4


### Exercice 2 - Planning : identifier puis relâcher la coupable

Encodez un planning sur `j` : (a) `j >= 8`, (b) `j <= 18`, (c) `j != 3`, (d) `j == 3`. Obtenez le core, identifiez la coupable, puis **relâchez-la** (re-vérifiez le sous-ensemble sans elle) -> le système devient SATISFIABLE.

**Étapes** : core = {j_ne_3, j_eq_3} ; retirer l'une des deux du solveur ; re-`check()` -> `sat`.

In [9]:
// Exercice 2 : planning, identifier puis relâcher la contrainte coupable
// TODO etudiant :
// Etape 1 : encoder (a)(b)(c)(d) sur j, obtenir le core
// Etape 2 : core = {j_ne_3, j_eq_3} -> choisir laquelle relâcher
// Etape 3 : re-check sur le sous-ensemble SANS la coupable -> SATISFIABLE
// Indice : reconstruisez un second `Solver` avec les deux contraintes restantes.
let result: Option<u32> = None;  // TODO etudiant
println!("Exercice 2 a completer : identifiez et relachez la contrainte coupable");
let _ = &result;

Exercice 2 a completer : identifiez et relachez la contrainte coupable


### Exercice 3 - Core minimal parmi 10 contraintes

Construisez 10 contraintes sur `k` : 3 conflictuelles (`k == 1`, `k == 2`, `k == 3`) + 7 compatibles (`k >= 0`, `k <= 100`, etc.). Le core doit valoir exactement **3** (les 3 incompatibles), les 7 compatibles étant écartées.

**Étapes** : 10 `assert_and_track` ; `check()` ; compter `get_unsat_core().len()` -> attendu 3.

In [10]:
// Exercice 3 : core minimal parmi 10 contraintes
// TODO etudiant :
// Etape 1 : k = Int::new_const("k") ; 10 contraintes (3 k==n incompatibles + 7 compatibles)
// Etape 2 : s.assert_and_track pour les 10
// Etape 3 : s.check() + s.get_unsat_core().len() -> doit valoir 3
let nb_coupables: Option<u32> = None;  // TODO etudiant
println!("Exercice 3 a completer : mesurer |core| parmi 10 contraintes (attendu : 3)");
let _ = &nb_coupables;

Exercice 3 a completer : mesurer |core| parmi 10 contraintes (attendu : 3)


## Conclusion

Le UNSAT core transforme le solveur d'oracle binaire en **outil de diagnostic**. Là où `unsat` dit seulement « impossible », `get_unsat_core()` dit « impossible **à cause de ces contraintes-là** » - le sous-ensemble minimal responsable du conflit. Sur une spécification surcontrainte, ce diagnostic est directement actionnable : il pointe la contrainte à relâcher.

Cette série Z3-Python (carnets 01 à 13) couvre maintenant les **trois postures** du solveur : **décider** (sat/unsat, 01-11), **optimiser** (Optimize, 06/08), et ici **expliquer** (unsat_core, 13). Là où l'ingénieur classique debug à l'aveugle un système insatisfiable, le solveur SMT isole chirurgicalement la cause - la différence entre un oracle et un partenaire de raisonnement.

Le compagnon [Z3-13b - le MUS](Z3-13b-UnsatCores-MUS-Python.ipynb) approfondit ce langage : core invisible à l'inspection, minimalité non garantie par Z3, et l'algorithme deletion-based qui calcule le sous-ensemble irréductible.

***

> Carnet issu de l'EPIC #1206 ; portage Rust du lot #27129 (parité Rust, troisième langage des jumeaux).